# AdaCLIP fixed-perturbation evaluation
Enable a GPU and Internet, then attach MVTec AD, VisA, and the generated perturbation dataset. The notebook clones this evaluator and official AdaCLIP automatically, discovers the mounted paths, downloads and checksum-verifies the released AdaCLIP weights from the authors' HuggingFace Space, inventories both prompt folders, and evaluates every matching selected setup.

AdaCLIP evaluates zero-shot: MVTec uses the VisA/ClinicDB-trained weights and VisA uses the MVTec/ColonDB-trained weights, exactly as `test.sh` does. Its official `test.py` supports batch size 1 only, so the adapter always runs one image at a time regardless of `BATCH_SIZE`.


In [ ]:
# ============================== USER SELECTION ==============================
# Target evaluation datasets: any non-empty subset of ('mvtec', 'visa').
TARGETS = ('mvtec', 'visa')
# Attack-generation sources: ('mvtec',), ('visa',), both, or None for every source in the manifests.
SOURCE_DATASETS = ('mvtec',)
# Explicit mounted MVTec root, or None to auto-discover exactly one valid MVTec dataset.
MVTEC_INPUT = '/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'
# Explicit mounted VisA root, or None to auto-discover exactly one valid VisA dataset.
VISA_INPUT = '/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'
# Perturbation dataset root containing setups/, or None to auto-discover exactly one.
PERTURBATION_INPUT = '/kaggle/input/datasets/parsaorbot/perterbation-generated'
# Prompt families: ('frozen_prompt',), ('learnable_prompt',), both, or None for every mounted family.
PROMPT_MODES = ('frozen_prompt', 'learnable_prompt')
# Setup IDs: tuple of selected base IDs, or None for every mounted setup.
# Valid base IDs: steps{500|800}_eps{2|4}, optionally followed by _margin_topk,
# then optionally _gradnorm (gradient-normalized combined loss; those setups
# provide the 'combined' loss mode only).
SETUP_IDS = None
# Attack scopes: any non-empty subset of ('per_dataset', 'cross_dataset',
# 'per_category', 'per_image'). per_dataset covers the source dataset itself;
# cross_dataset delivers the same delta to the other dataset.
SCOPES = ('per_dataset', 'cross_dataset', 'per_category', 'per_image')
# Category names, e.g. ('bottle',), or None for all; selecting categories excludes per-dataset rows.
CATEGORIES = None
# Attack directions: ('normal_to_abnormal',), ('abnormal_to_normal',), both, or None for both.
DIRECTIONS = None
# Objective modes: ('global',), ('local',), ('combined',), any combination, or None for all.
LOSS_MODES = None
# Loss families: ('ce_focal_dice',), ('margin_topk',), both, or None for all available.
LOSS_FORMULATIONS = None
# Loader batch size only. AdaCLIP's official text-prompt layer is not batch-safe,
# so the adapter forwards one image at a time whatever this is set to.
BATCH_SIZE = 1
# True saves compressed score/map NPZ files; False keeps only metrics and qualitative samples.
SAVE_PREDICTIONS = False
# True replaces an existing completed model result; False prevents accidental replacement.
OVERWRITE = False
# Positive integer limits conditions for a smoke test; None evaluates every selected condition.
MAX_CONDITIONS = None# Cap how many conditions keep qualitative samples, applied per scope and
# ranked by targeted attack success rate; None keeps samples for every
# condition. Sample volume grows with condition count, so per_category and
# per_image need this far more than the dataset-level scopes do.
MAX_SAMPLE_CONDITIONS = 25
# Which pixel-threshold mode the sample images are rendered at. Must be a
# subset of the evaluated modes; None renders every one, which multiplies the
# sample output. 'clean_pixel_f1' is the calibrated pixel F1-max threshold and
# is the right one to look at; 'image_f1' is the IMAGE F1 threshold applied to
# pixels as a deliberate ablation. All three are still scored in the CSVs
# whatever this is set to.
QUALITATIVE_THRESHOLD_MODES = ('clean_pixel_f1',)


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

WORKING = Path('/kaggle/working')
EVALUATOR_ROOT = WORKING / 'adversarial-perturbation-evaluator'
ADACLIP_ROOT = WORKING / 'AdaCLIP'
OUTPUT_ROOT = WORKING / 'fixed_perturbation_results'
EVALUATOR_URL = 'https://github.com/Parsagh05/adversarial-perturbation-evaluator.git'
ADACLIP_URL = 'https://github.com/caoyunkang/AdaCLIP.git'

def clone_or_update(url, destination):
    if (destination / '.git').is_dir():
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(destination)], check=True)

clone_or_update(EVALUATOR_URL, EVALUATOR_ROOT)
clone_or_update(ADACLIP_URL, ADACLIP_ROOT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    f'{EVALUATOR_ROOT}[adaclip]',
], check=True)
print('Evaluator commit:', subprocess.check_output(
    ['git', '-C', str(EVALUATOR_ROOT), 'rev-parse', 'HEAD'], text=True
).strip())

In [ ]:
from fpeval.kaggle import discover_kaggle_inputs, inventory_attack_setups

mounted = discover_kaggle_inputs(
    '/kaggle/input',
    mvtec_root=MVTEC_INPUT,
    visa_root=VISA_INPUT,
    attacks_root=PERTURBATION_INPUT,
)
inventory = inventory_attack_setups(mounted.attacks_root)
print('MVTec root:', mounted.mvtec_root)
print('VisA root:', mounted.visa_root)
print('Perturbation root:', mounted.attacks_root)
print('Available attack setups:', json.dumps(inventory, indent=2))

selected_modes = tuple(PROMPT_MODES) if PROMPT_MODES is not None else tuple(inventory)
available_selected_modes = tuple(mode for mode in selected_modes if mode in inventory)
if not available_selected_modes:
    raise RuntimeError(f'None of the selected prompt modes are available: {selected_modes}')
if SETUP_IDS is not None:
    available_ids = {setup_id for mode in available_selected_modes for setup_id in inventory[mode]}
    missing_ids = sorted(set(SETUP_IDS) - available_ids)
    if missing_ids:
        raise RuntimeError(f'Selected setup IDs are not mounted: {missing_ids}')
print('Selected prompt modes present on disk:', available_selected_modes)
print('Selected setup IDs:', SETUP_IDS if SETUP_IDS is not None else 'all available')

In [ ]:
# The released weights live on the authors' HuggingFace Space; the README links
# Google Drive only, which cannot be fetched unattended. resolve_checkpoint
# downloads once, verifies sha256, and reuses the cache on later runs.
from fpeval.adapters.adaclip import ZERO_SHOT_CHECKPOINT, resolve_checkpoint

WEIGHTS_ROOT = WORKING / 'adaclip-weights'
resolved_checkpoints = {}
for target in TARGETS:
    name = ZERO_SHOT_CHECKPOINT[target]
    path = resolve_checkpoint(name, download_root=WEIGHTS_ROOT)
    resolved_checkpoints[target] = path
    print(f'{target}: {name} -> {path} ({path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
all_model_kwargs = {
    target: {
        'repository': str(ADACLIP_ROOT),
        'target_dataset': target,
        'checkpoint': str(resolved_checkpoints[target]),
        'download_root': str(WEIGHTS_ROOT),
    }
    for target in TARGETS
}
config = {
    'attacks_root': str(mounted.attacks_root),
    'output_root': str(OUTPUT_ROOT),
    'model': 'adaclip',
    'mvtec_root': str(mounted.mvtec_root),
    'visa_root': str(mounted.visa_root),
    'targets': list(TARGETS),
    'source_datasets': list(SOURCE_DATASETS) if SOURCE_DATASETS is not None else None,
    'scopes': list(SCOPES),
    'prompt_modes': list(available_selected_modes),
    'setup_ids': list(SETUP_IDS) if SETUP_IDS is not None else None,
    'categories': list(CATEGORIES) if CATEGORIES is not None else None,
    'directions': list(DIRECTIONS) if DIRECTIONS is not None else None,
    'loss_modes': list(LOSS_MODES) if LOSS_MODES is not None else None,
    'loss_formulations': list(LOSS_FORMULATIONS) if LOSS_FORMULATIONS is not None else None,
    'model_kwargs_by_target': {target: all_model_kwargs[target] for target in TARGETS},
    'device': 'cuda',
    'batch_size': BATCH_SIZE,
    'image_size': 518,
    # AdaCLIP returns full-resolution maps and the official evaluation applies
    # gaussian_filter(sigma=4) to them, so the shared evaluator does the same.
    'gaussian_sigma': 4.0,
    'pixel_threshold_modes': ['fixed_0_5', 'image_f1', 'clean_pixel_f1'],
    'verify_checksums': True,
    'save_predictions': SAVE_PREDICTIONS,
    'save_qualitative_samples': True,
    'write_separated_results': True,
    'create_output_archives': True,
    'overwrite': OVERWRITE,
    'max_conditions': MAX_CONDITIONS,
    'max_sample_conditions': MAX_SAMPLE_CONDITIONS,
    'qualitative_threshold_modes': list(QUALITATIVE_THRESHOLD_MODES)
    if QUALITATIVE_THRESHOLD_MODES is not None else None,
}
config_path = WORKING / 'adaclip.json'
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(config_path.read_text())

In [ ]:
subprocess.run([sys.executable, '-m', 'fpeval', '--config', str(config_path)], check=True)
model_output = OUTPUT_ROOT / 'adaclip'
threshold_path = model_output / 'thresholds.json'
print('Automatically calibrated and frozen thresholds:', threshold_path)
print(threshold_path.read_text()[:4000])
print('Downloadable output archives:')
for name in ('adaclip', 'adaclip_separated', 'adaclip_samples', 'adaclip_samples_separated'):
    print(' -', OUTPUT_ROOT / f'{name}.zip')